# Icechunk + Zarr + RAPIDS: a versioned single-cell GPU workflow

This notebook walks through one clean, linear workflow that shows how the three tools fit together:

- **Zarr (v3)** is the on-disk array format — the CSR expression matrix, `obs`/`var`, layers, and analysis results all live as Zarr arrays/groups, written so the store stays **anndata-readable**.
- **Icechunk** wraps that Zarr store with **git-like versioning** — every step is a `session` of staged writes made durable by a `commit`, and work happens on **branches** so unchanged data (e.g. raw counts) is shared across snapshots with zero copy.
- **RAPIDS (rapids-singlecell)** runs the actual single-cell compute on the **GPU**, streaming Zarr chunks straight onto the device.

The story:

| Step | What happens |
|------|--------------|
| **0 — Ingest** | Create the repo, copy raw CSR counts + `obs`/`var` onto `main`. |
| **1 — Normalize** | Branch, normalize + log1p **all** cells on multi-GPU dask, write to a **new** `layers/norm` — raw `X` is untouched. Commit, fast-forward `main`. |
| **2 — Subset cluster** | Branch, pull **one cell type** from `layers/norm` to a single GPU, run HVG → PCA → neighbors → UMAP → Leiden, commit results into a scoped `analyses/` group. |
| **3 — Inspect** | Open `main` and show the final layout + commit history. |

> **GPU/CUDA only** — this runs on an HPC/GPU node (4 GPUs assumed). `rapids-singlecell` is not installed in the local CPU env.

## Imports

In [ ]:
from __future__ import annotations

import sys
import time
import warnings

import dask
dask.config.set({"distributed.scheduler.worker-ttl": None})

from dask_cuda import LocalCUDACluster
from dask.distributed import Client
import cupy as cp

import numpy as np
import scipy.sparse as sp
import zarr
import anndata as ad
from anndata.io import read_elem, write_elem
from anndata.experimental import read_elem_lazy

from icechunk import Repository, local_filesystem_storage

import rapids_singlecell as rsc

## Configuration

One place for the source store, repo path, the cell type we'll cluster in Step 2, and the parallelism knobs.

In [ ]:
SOURCE_ZARR = "/home/workspace/zarrs/13M_50M_pbmc_soundlife.zarr"  # existing anndata CSR zarr (raw counts)
REPO_PATH   = "/home/workspace/zarrs/demo_repo"                    # icechunk repo (fresh each run)

CELL_TYPE_COL = "predicted_AIFI_L1"
CELL_TYPE_VAL = "B cell"

SPARSE_CHUNK_SIZE = 5_000   # rows per dask block when streaming the CSR X
N_TOP_GENES       = 2000
RANDOM_SEED       = 5671

# zarr read parallelism: async.concurrency dispatches chunk fetches; threading.max_workers
# sizes the decode thread pool. blosc kept single-threaded so it doesn't fight the pool.
ZARR_ASYNC_CONCURRENCY = 60
ZARR_CODEC_WORKERS     = 60
BLOSC_NTHREADS         = 1

## Helper functions

Small utilities reused across the steps:

- `_log` — timestamped, flushed progress to stderr.
- `_copy_csr` — stream-copy a CSR group (`data`/`indices`/`indptr`) chunk-by-chunk, re-stamping the anndata encoding attrs so the result round-trips.
- `_load_adata` — open a Zarr/Icechunk store as a **lazy** AnnData (dask-backed `X`) so the matrix is streamed to the GPU, never fully materialized on the host.
- `_canon_csr` — rebuild a clean, C-contiguous, `int32`-indexed, `float32` CSR for a subset before it goes to the GPU (cupyx kernels only bind contiguous int32/int64 layouts).

In [ ]:
def _log(msg):
    """Timestamped, flushed progress line (block-buffered stdout looks 'hung')."""
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True, file=sys.stderr)


def _copy_csr(src_grp, dst_parent, name):
    """Stream-copy a csr group (data/indices/indptr 1D arrays) into dst_parent[name]."""
    dst = dst_parent.require_group(name)
    dst.attrs["encoding-type"] = src_grp.attrs["encoding-type"]
    dst.attrs["encoding-version"] = "0.1.0"
    dst.attrs["shape"] = list(src_grp.attrs["shape"])
    for child in ("data", "indices", "indptr"):
        s = src_grp[child]
        d = dst.require_array(child, shape=s.shape, dtype=s.dtype,
                              chunks=s.chunks, overwrite=True)
        d.attrs["encoding-type"] = "array"
        d.attrs["encoding-version"] = "0.2.0"
        step = s.chunks[0] * 16
        n = s.shape[0]
        _log(f"    _copy_csr/{name}/{child}: {n:,} elems ({s.dtype})")
        for i in range(0, n, step):
            d[i:i + step] = s[i:i + step]


def _load_adata(store, x_key="X"):
    """Lazy AnnData (dask X) backed by a zarr/icechunk store; x_key may be nested."""
    f = zarr.open_group(store=store, mode="r")
    node = f
    for part in x_key.split("/"):
        node = node[part]
    shape = tuple(node.attrs["shape"]) if "shape" in node.attrs else tuple(node.shape)
    X_dask = read_elem_lazy(node, (SPARSE_CHUNK_SIZE, shape[1]))
    if np.issubdtype(X_dask.dtype, np.integer):
        X_dask = X_dask.astype(np.float32)
    return ad.AnnData(X=X_dask, obs=read_elem(f["obs"]), var=read_elem(f["var"]))


def _canon_csr(m):
    """Clean, C-contiguous, int32-indexed, float32 CSR for a single-cell-type subset.

    The full store has nnz > 2**31 (int64 indptr/indices) and a dask mask-gather can
    leave blocks non-contiguous — both make cupyx CSR kernels miss their nanobind
    overloads. For one cell type every value fits int32, so rebuild one clean CSR.
    """
    m = m if sp.isspmatrix_csr(m) else sp.csr_matrix(m)
    return sp.csr_matrix(
        (np.ascontiguousarray(m.data, dtype=np.float32),
         np.ascontiguousarray(m.indices, dtype=np.int32),
         np.ascontiguousarray(m.indptr, dtype=np.int32)),
        shape=m.shape,
    )

## Environment setup: GPU cluster + memory pool + zarr parallelism

Spin up a 4-GPU `LocalCUDACluster`, point CuPy at RMM **managed** memory (so blocks can spill), and set the two zarr read knobs deliberately (see *silent performance killer #1* — don't let dask threads × codec threads × blosc threads multiply).

In [ ]:
# Multi-GPU dask cluster (4 devices). Each worker streams + decodes zarr blocks in place.
cluster = LocalCUDACluster(
    CUDA_VISIBLE_DEVICES="0,1,2,3",
    protocol="tcp",
    threads_per_worker=16,
    rmm_managed_memory=True,
    rmm_allocator_external_lib_list="cupy",
)
client = Client(cluster)
client

In [ ]:
# Client-side RMM pool for CuPy (managed memory so large gathers can oversubscribe).
import rmm
from rmm.allocators.cupy import rmm_cupy_allocator
rmm.reinitialize(managed_memory=True, pool_allocator=False, devices=[0, 1, 2, 3])
cp.cuda.set_allocator(rmm_cupy_allocator)

# zarr chunk-level parallelism + codec thread pool; blosc single-threaded under it.
zarr.config.set({"async.concurrency": ZARR_ASYNC_CONCURRENCY,
                 "threading.max_workers": ZARR_CODEC_WORKERS})
from numcodecs import blosc
blosc.set_nthreads(BLOSC_NTHREADS)

# anndata floods stderr with one autosharding notice per obs/var column; silence just that.
warnings.filterwarnings("ignore", message=".*autosharding will be the default.*",
                        category=UserWarning)

_log(f"parallelism: zarr async.concurrency={ZARR_ASYNC_CONCURRENCY} "
     f"threading.max_workers={ZARR_CODEC_WORKERS} blosc.nthreads={BLOSC_NTHREADS} "
     f"| zarr {zarr.__version__}")

## Step 0 — Create the repo and ingest raw counts onto `main`

Create a fresh Icechunk repository, open a **writable session** on `main`, stream-copy the source CSR `X` plus `obs`/`var`, and `commit`. After this, `main` holds the raw-count AnnData.

In [ ]:
import os, shutil

abs_repo = os.path.abspath(REPO_PATH)
shutil.rmtree(abs_repo, ignore_errors=True)   # clean prefix — icechunk only creates into one
storage = local_filesystem_storage(abs_repo)
repo = Repository.create(storage)
print("created fresh repo:", abs_repo)

In [ ]:
src = zarr.open_group(SOURCE_ZARR, mode="r")

session = repo.writable_session("main")
root = zarr.open_group(store=session.store, mode="a")
root.attrs["encoding-type"] = "anndata"
root.attrs["encoding-version"] = "0.1.0"

_copy_csr(src["X"], root, "X")
write_elem(root, "obs", read_elem(src["obs"]))
write_elem(root, "var", read_elem(src["var"]))

print("step0 ingest commit:", session.commit("ingest: raw counts"))

## Step 1 — User 1: normalize + log1p all cells → `layers/norm`

Branch `user1-normalize` off `main`, stream **every** cell through the GPU (`normalize_total` + `log1p`), and write the result to a **new** `layers/norm`. Raw counts in `X` are never touched — the new layer is the only thing added to the snapshot. Commit, then fast-forward `main` onto it.

Note `_load_adata` is given the **read-only** session store: it's picklable, so the lazy dask-X graph can ship to the GPU workers.

In [ ]:
_log("step1 user1_normalize: START")
repo.create_branch("user1-normalize", repo.lookup_branch("main"))
session = repo.writable_session("user1-normalize")
root = zarr.open_group(store=session.store, mode="a")

# Lazy AnnData over the read-only session store → streams to GPU workers.
adata = _load_adata(repo.readonly_session(branch="user1-normalize").store)
rsc.get.anndata_to_GPU(adata)
rsc.pp.calculate_qc_metrics(adata)
rsc.pp.normalize_total(adata)
rsc.pp.log1p(adata)

# Write normalized matrix to a NEW layer; X (raw counts) stays as-is.
layers = root.require_group("layers")
layers.attrs["encoding-type"] = "dict"
layers.attrs["encoding-version"] = "0.1.0"
write_elem(layers, "norm", adata.X)

snap = session.commit("user1: normalized+log1p in layers/norm; raw counts kept in X")
repo.reset_branch("main", snap)   # fast-forward main
print("step1 user1 commit:", snap)

## Step 2 — User 2: cluster one cell type from `layers/norm`

Branch `user2-subset` off `main`. Read `layers/norm` lazily, materialize **only** the one cell type to the host as a clean CSR, then run the whole clustering pipeline in-memory on a **single** GPU: HVG → scale → PCA → neighbors → UMAP → Leiden. The big `X`/`layers` arrays are never rewritten — results land in a scoped `analyses/<cell_type>` group.

In [ ]:
repo.create_branch("user2-subset", repo.lookup_branch("main"))
session = repo.writable_session("user2-subset")
root = zarr.open_group(store=session.store, mode="a")

# Source X from layers/norm; materialize ONLY this cell type to host as one clean CSR.
adata = _load_adata(repo.readonly_session(branch="user2-subset").store, x_key="layers/norm")
mask = (adata.obs[CELL_TYPE_COL] == CELL_TYPE_VAL).to_numpy()
idx = np.where(mask)[0]
_log(f"  {CELL_TYPE_VAL}: {idx.size:,} cells; materializing subset to host")

sub = adata[mask]
adata = ad.AnnData(X=_canon_csr(sub.X.compute()), obs=sub.obs.copy(), var=sub.var.copy())

In [ ]:
# Whole subset pipeline in-memory on a single GPU.
rsc.get.anndata_to_GPU(adata)
rsc.pp.highly_variable_genes(adata, flavor="seurat", n_top_genes=N_TOP_GENES)
hvg_mask = np.asarray(adata.var["highly_variable"])
adata = adata[:, hvg_mask].copy()

adata.X = adata.X.astype("float64")  # covariance_eigh PCA is happier in float64
rsc.pp.scale(adata, max_value=10, zero_center=False)
rsc.pp.pca(adata, n_comps=30, svd_solver="covariance_eigh", random_state=RANDOM_SEED)
rsc.pp.neighbors(adata, n_neighbors=20, n_pcs=30,
                 algorithm="mg_ivfflat", random_state=RANDOM_SEED)
rsc.tl.umap(adata, min_dist=0.45, init_pos="spectral",
            n_components=2, random_state=RANDOM_SEED)
rsc.tl.leiden(adata, resolution=1.1, n_iterations=100, random_state=RANDOM_SEED)
rsc.get.anndata_to_CPU(adata)

In [ ]:
# Commit results into a scoped group — X/layers untouched.
g = root.require_group("analyses").require_group(CELL_TYPE_VAL)
write_elem(g, "cell_index", idx.astype("int64"))
write_elem(g, "leiden", np.asarray(adata.obs["leiden"].astype(str)))
write_elem(g, "X_umap", np.asarray(adata.obsm["X_umap"]))
write_elem(g, "highly_variable", hvg_mask)

snap = session.commit(f"user2: leiden/umap/hvg for {CELL_TYPE_VAL} subset ({idx.size} cells)")
repo.reset_branch("main", snap)
print("step2 user2 commit:", snap)

## Step 3 — Inspect the final `main`

Open a read-only snapshot of `main` and show the final layout (raw `X`, `layers/norm`, the scoped `analyses/` group) and the full commit history — every step is one snapshot, with unchanged data shared across them.

In [ ]:
root = zarr.open_group(store=repo.readonly_session(branch="main").store, mode="r")
print("final main root keys:", list(root.keys()))
print("   X shape (raw counts):     ", tuple(root["X"].attrs["shape"]))
print("   layers/norm shape (norm): ", tuple(root["layers"]["norm"].attrs["shape"]))

a = root["analyses"][CELL_TYPE_VAL]
print(f"   analyses/{CELL_TYPE_VAL}:", {k: a[k].shape for k in a.keys()})

print("\n   history:")
for info in repo.ancestry(branch="main"):
    print("     ", info.id, info.message)

## TEAR IT DOWN

In [ ]:
client.shutdown()